# pheatmap tutorial (Python port)

Python port of `man/pheatmap_tutorial.Rmd` from the upstream R package.  The function `pheatmap.pheatmap` draws clustered heatmaps with control over cell size, clustering, scaling, annotations, gaps, and legend formatting.

## Setup

In [1]:
import numpy as np
import pandas as pd
from pheatmap import pheatmap
from pheatmap._colours import colour_ramp_palette

rng = np.random.default_rng(1)

## Create test matrix

The same construction as the R tutorial; the values will differ because NumPy's `default_rng(1)` is not the same stream as R's `set.seed(1)`, but the block structure is identical.

In [2]:
test = rng.standard_normal((20, 10))
test[0:10, 0::2] += 3
test[10:20, 1::2] += 2
test[14:20, 1::2] += 4
test = pd.DataFrame(
    test,
    index=[f"Gene{i+1}" for i in range(20)],
    columns=[f"Test{i+1}" for i in range(10)],
)
test.head()

,Test1,Test2,Test3,Test4,Test5,Test6,Test7,Test8,Test9,Test10
Gene1,3.345584,0.821618,3.330437,-1.303157,3.905356,0.446375,2.463047,0.581118,3.364572,0.294132
Gene2,3.028422,0.546713,2.263546,-0.162910,2.517881,0.598846,3.039722,-0.292457,2.218092,-0.257192
Gene3,3.008142,-0.275603,4.294064,1.006724,0.288838,-1.889013,2.825228,-0.422190,3.213643,0.217322
Gene4,5.117839,-1.112021,2.622395,2.042772,3.646703,0.663063,2.485994,-1.648075,3.167465,0.109014
Gene5,1.772648,-0.683227,2.927956,-0.944752,2.901730,0.095483,3.035586,-0.506292,3.593748,0.891167


## Draw heatmaps

In [3]:
pheatmap(test)

In [4]:
pheatmap(test, kmeans_k=2)

In [5]:
pheatmap(test, scale="row", clustering_distance_rows="correlation")

In [6]:
pheatmap(
    test,
    color=colour_ramp_palette(["navy", "white", "#CD2626"])(50),
)

In [7]:
pheatmap(test, cluster_rows=False)

In [8]:
pheatmap(test, legend=False)

## Show text within cells

In [9]:
pheatmap(test, display_numbers=True)

In [10]:
pheatmap(test, display_numbers=True, number_format="%.1e")

In [11]:
stars = np.where(test.values > 5, "*", "")
pheatmap(test, display_numbers=stars)

In [12]:
pheatmap(
    test,
    cluster_rows=False,
    legend_breaks=list(range(-1, 5)),
    legend_labels=["0", "1e-4", "1e-3", "1e-2", "1e-1", "1"],
)

## Fix cell sizes

In [13]:
pheatmap(test, cellwidth=15, cellheight=12, main="Example heatmap")

To save to a file, pass `filename="out.pdf"` (also supports `.png`, `.jpeg`, `.tiff`, `.bmp`).

## Generate annotations

In [14]:
annotation_col = pd.DataFrame(
    {
        "CellType": pd.Categorical([f"CT{1 + (i % 2)}" for i in range(10)]),
        "Time": list(range(1, 11)),
    },
    index=[f"Test{i+1}" for i in range(10)],
)
annotation_row = pd.DataFrame(
    {"GeneClass": pd.Categorical(["Path1"] * 10 + ["Path2"] * 4 + ["Path3"] * 6)},
    index=[f"Gene{i+1}" for i in range(20)],
)

In [15]:
pheatmap(test, annotation_col=annotation_col)

In [16]:
pheatmap(test, annotation_col=annotation_col, annotation_legend=False)

In [17]:
pheatmap(test, annotation_col=annotation_col, annotation_row=annotation_row)

## Change angle of column labels

In [18]:
pheatmap(
    test,
    annotation_col=annotation_col,
    annotation_row=annotation_row,
    angle_col="45",
)

In [19]:
pheatmap(test, annotation_col=annotation_col, angle_col="0")

## Specify colors

In [20]:
ann_colors = {
    "Time": ["white", "firebrick"],
    "CellType": {"CT1": "#1B9E77", "CT2": "#D95F02"},
    "GeneClass": {"Path1": "#7570B3", "Path2": "#E7298A", "Path3": "#66A61E"},
}

In [21]:
pheatmap(
    test,
    annotation_col=annotation_col,
    annotation_colors=ann_colors,
    main="Title",
)

In [22]:
pheatmap(
    test,
    annotation_col=annotation_col,
    annotation_row=annotation_row,
    annotation_colors=ann_colors,
)

In [23]:
pheatmap(
    test,
    annotation_col=annotation_col,
    annotation_colors={"CellType": ann_colors["CellType"]},
)

## Gaps in heatmaps

In [24]:
pheatmap(
    test,
    annotation_col=annotation_col,
    cluster_rows=False,
    gaps_row=[10, 14],
)

In [25]:
pheatmap(
    test,
    annotation_col=annotation_col,
    cluster_rows=False,
    gaps_row=[10, 14],
    cutree_cols=2,
)

## Custom strings as row/col names

In [26]:
labels_row = [""] * 17 + ["Il10", "Il15", "Il1b"]
pheatmap(test, annotation_col=annotation_col, labels_row=labels_row)

## Clustering from a pre-computed distance matrix

In [27]:
from scipy.spatial.distance import pdist

drows = pdist(test.values, metric="euclidean")
dcols = pdist(test.values.T, metric="euclidean")
pheatmap(test, clustering_distance_rows=drows, clustering_distance_cols=dcols)

## Modify cluster ordering with a clustering callback

The callback receives `(hc, mat)` and returns a modified `HClust`.

In [28]:
from pheatmap._cluster import _hclust_from_linkage
from scipy.cluster.hierarchy import leaves_list

def callback(hc, mat):
    # Reorder leaves by the first right-singular vector of mat.T.
    _, _, vt = np.linalg.svd(mat.T, full_matrices=False)
    sv = vt[0]
    order = np.argsort(sv)
    return _hclust_from_linkage(hc.linkage, labels=hc.labels)

pheatmap(test, clustering_callback=callback)